# Retrieval: implement it, then break it

Six exercises from **LLM Engineering Lab**, maintained by Rui Wang.

This notebook is self-contained: no API key, package install, model download, or cloned repository is required. All sparse weights and token-similarity matrices are constructed. These are mechanics exercises, not a benchmark of learned retrievers.

Run the two collapsed setup cells and the checker below. Each exercise cell contains a starter function. Replace its `NotImplementedError`, rerun that cell, then run its check. `TODO` means unfinished; `PASS` means the visible contract tests passed. Passing tests is not proof of full correctness.

The setup includes reference answers so the notebook can run without fetching files. Leave it collapsed while solving. Reference code is available to inspect afterward.

In [ ]:
"""Reference scoring functions, not trained neural retrievers.

Sparse weights and token similarities are supplied by the caller. No model,
network connection, API key, GPU, or third-party package is used.
"""
from __future__ import annotations

from collections import Counter, defaultdict
from math import isfinite, log1p
from typing import Mapping, Sequence


def bm25(query: Sequence[str], document: Sequence[str],
         corpus: Sequence[Sequence[str]], *, k1: float = 1.2,
         b: float = 0.75) -> float:
    """Conventional BM25. The document must belong to the supplied corpus."""
    if not corpus or not isfinite(k1) or k1 < 0 or not 0 <= b <= 1:
        raise ValueError("Need a nonempty corpus, finite k1 >= 0, and 0 <= b <= 1")
    if list(document) not in [list(d) for d in corpus]:
        raise ValueError("The scored document must belong to corpus")
    avg_length = sum(map(len, corpus)) / len(corpus)
    if avg_length == 0:
        return 0.0
    frequencies = Counter(document)
    result = 0.0
    for term in set(query):
        tf = frequencies[term]
        if tf == 0:
            continue
        df = sum(term in d for d in corpus)
        idf = log1p((len(corpus) - df + 0.5) / (df + 0.5))
        length = 1 - b + b * len(document) / avg_length
        result += idf * tf * (k1 + 1) / (tf + k1 * length)
    return result


def sparse_dot(query: Mapping[str, float], doc: Mapping[str, float]) -> float:
    return sum(weight * doc.get(term, 0.0) for term, weight in query.items())


def sparse_index_scores(query: Mapping[str, float],
                        documents: Mapping[str, Mapping[str, float]]) -> dict[str, float]:
    """Execute the same sparse dot product by traversing inverted postings."""
    postings: dict[str, list[tuple[str, float]]] = defaultdict(list)
    for doc_id, vector in documents.items():
        for term, weight in vector.items():
            if weight:
                postings[term].append((doc_id, weight))
    scores = dict.fromkeys(documents, 0.0)
    for term, query_weight in query.items():
        for doc_id, doc_weight in postings.get(term, []):
            scores[doc_id] += query_weight * doc_weight
    return scores


def maxsim(matrix: Sequence[Sequence[float]], *,
           valid_doc: Sequence[bool] | None = None) -> float:
    """Sum row maxima: rows=query vectors, columns=document vectors.

    Mask columns BEFORE taking maxima. Reject an empty query/document instead
    of inventing a score. ColBERT query-augmentation vectors are not padding to
    discard; model-specific query masking is intentionally outside this lab.
    """
    if not matrix or not matrix[0]:
        raise ValueError("Need at least one query row and document column")
    n = len(matrix[0])
    if any(len(row) != n for row in matrix):
        raise ValueError("Ragged similarity matrix")
    if any(not isfinite(v) for row in matrix for v in row):
        raise ValueError("Similarity values must be finite")
    mask = list(valid_doc) if valid_doc is not None else [True] * n
    if len(mask) != n or not any(mask):
        raise ValueError("Need one mask value per column and a valid document token")
    return sum(max(v for v, keep in zip(row, mask) if keep) for row in matrix)


def rrf(rankings: Sequence[Sequence[str]], *, c: float = 60.0) -> list[tuple[str, float]]:
    """Fuse truncated, one-based ranked lists; missing documents contribute zero."""
    if not isfinite(c) or c < 0:
        raise ValueError("c must be finite and nonnegative")
    scores: dict[str, float] = defaultdict(float)
    for ranking in rankings:
        if len(set(ranking)) != len(ranking):
            raise ValueError("Deduplicate each input ranking before fusion")
        for rank, doc_id in enumerate(ranking, 1):
            scores[doc_id] += 1 / (c + rank)
    return sorted(scores.items(), key=lambda pair: (-pair[1], pair[0]))



def tf_weight(tf: int, length: int, avg_length: float, *,
              k1: float = 1.2, b: float = 0.75) -> float:
    """BM25's length-adjusted TF factor, excluding IDF.

    Require integer 0 <= tf <= length, avg_length > 0, finite k1 >= 0,
    and finite 0 <= b <= 1. Empty documents score zero. This example
    uses analyzed-token lengths, not characters or bytes.
    """
    if (not isinstance(tf, int) or isinstance(tf, bool)
            or not isinstance(length, int) or isinstance(length, bool)
            or tf < 0 or length < tf
            or not isfinite(avg_length) or avg_length <= 0
            or not isfinite(k1) or k1 < 0
            or not isfinite(b) or not 0 <= b <= 1):
        raise ValueError("Invalid TF, length, average length, k1, or b")
    if tf == 0:
        return 0.0
    correction = 1 - b + b * length / avg_length
    return tf * (k1 + 1) / (tf + k1 * correction)


def recall_at_k(relevant: set[str], ranking: Sequence[str], k: int) -> float:
    """Fraction of relevant IDs found among the first k positions.

    Duplicate results occupy positions but count once. For an empty relevant
    set, recall is undefined: raise ValueError rather than invent a perfect
    or zero score. Evaluate unanswerable queries with a separate metric.
    """
    if not relevant:
        raise ValueError("Recall requires at least one relevant document")
    if not isinstance(k, int) or isinstance(k, bool) or k < 0:
        raise ValueError("k must be a nonnegative integer")
    return len(relevant.intersection(ranking[:k])) / len(relevant)


def candidate_union(matrices: Mapping[str, Sequence[Sequence[float]]], *,
                    per_query: int = 1) -> set[str]:
    """Union the best documents for each query-vector row.

    This is an exact toy screening rule, NOT PLAID or an ANN implementation.
    Each document first contributes its best token score for that query row;
    choose per_query documents, with alphabetical IDs breaking ties. Then
    union their IDs. All matrices have the same nonzero number of query rows.
    Values must be finite; columns need not have equal counts across docs.
    """
    if not isinstance(per_query, int) or isinstance(per_query, bool) or per_query <= 0:
        raise ValueError("per_query must be a positive integer")
    if not matrices:
        return set()
    rows = len(next(iter(matrices.values())))
    for matrix in matrices.values():
        maxsim(matrix)  # Validate nonempty rectangular matrices and finite values.
        if len(matrix) != rows:
            raise ValueError("All documents need the same query-row count")
    chosen: set[str] = set()
    for i in range(rows):
        ordered = sorted(matrices, key=lambda name: (-max(matrices[name][i]), name))
        chosen.update(ordered[:per_query])
    return chosen

from types import SimpleNamespace
REFERENCE = SimpleNamespace(tf_weight=tf_weight, sparse_dot=sparse_dot, maxsim=maxsim, rrf=rrf, recall_at_k=recall_at_k, candidate_union=candidate_union)


In [ ]:
"""Explicit expected values: reusable for both the reference and learner code."""
import unittest
# REFERENCE is defined by the previous cell.


class RetrievalChecks(unittest.TestCase):
    api = REFERENCE

    def test_tf_weight_average_length(self):
        self.assertAlmostEqual(self.api.tf_weight(1, 10, 10), 1.0)
        self.assertAlmostEqual(self.api.tf_weight(5, 10, 10), 11 / 6.2)

    def test_tf_weight_length_adjustment(self):
        self.assertAlmostEqual(self.api.tf_weight(1, 20, 10), 2.2 / 3.1)
        self.assertAlmostEqual(self.api.tf_weight(1, 5, 10), 2.2 / 1.75)

    def test_tf_weight_no_length_adjustment(self):
        self.assertEqual(self.api.tf_weight(1, 10, 10, b=0),
                         self.api.tf_weight(1, 100, 10, b=0))

    def test_tf_weight_zero(self):
        self.assertEqual(self.api.tf_weight(0, 0, 10, k1=0), 0.0)
        self.assertEqual(self.api.tf_weight(2, 10, 10, k1=0), 1.0)

    def test_tf_weight_invalid(self):
        for args in [(-1, 10, 10), (11, 10, 10), (1, 10, 0),
                     (1, 10, float("inf")), (1.5, 10, 10), (True, 10, 10)]:
            with self.subTest(args=args), self.assertRaises(ValueError):
                self.api.tf_weight(*args)
        for kwargs in [{"k1": -1}, {"k1": float("nan")}, {"b": 2}]:
            with self.subTest(kwargs=kwargs), self.assertRaises(ValueError):
                self.api.tf_weight(1, 10, 10, **kwargs)

    def test_sparse_dot_shared_keys(self):
        q = {"car": 1.5, "automobile": .7, "repair": 1.3, "maintenance": .9}
        d = {"automobile": 1.4, "repair": .8, "maintenance": 1.2}
        self.assertAlmostEqual(self.api.sparse_dot(q, d), 3.1)

    def test_sparse_dot_no_cross_coordinate_similarity(self):
        self.assertEqual(self.api.sparse_dot({"car": 2}, {"automobile": 3}), 0)

    def test_sparse_dot_empty(self):
        self.assertEqual(self.api.sparse_dot({}, {}), 0)
        self.assertEqual(self.api.sparse_dot({}, {"car": 1}), 0)

    def test_sparse_dot_commutative(self):
        a, b = {"x": 2, "y": 4}, {"y": 3, "z": 5}
        self.assertEqual(self.api.sparse_dot(a, b), 12)
        self.assertEqual(self.api.sparse_dot(b, a), 12)

    def test_maxsim_axes(self):
        self.assertAlmostEqual(self.api.maxsim([[.9, .1, 0], [.1, .8, .2]]), 1.7)

    def test_maxsim_padding(self):
        self.assertAlmostEqual(self.api.maxsim([[-.2, -.5, 0]],
                                              valid_doc=[True, True, False]), -.2)

    def test_maxsim_duplicate_columns(self):
        self.assertEqual(self.api.maxsim([[.9, .1], [.1, .8]]),
                         self.api.maxsim([[.9, .1, .9], [.1, .8, .1]]))

    def test_maxsim_not_boolean_and(self):
        self.assertGreater(self.api.maxsim([[1], [1], [.1]]),
                           self.api.maxsim([[.65], [.65], [.65]]))

    def test_maxsim_invalid_matrix(self):
        for matrix in [[], [[]], [[1], [2, 3]], [[float("nan")]], [[float("inf")]]]:
            with self.subTest(matrix=matrix), self.assertRaises(ValueError):
                self.api.maxsim(matrix)

    def test_maxsim_invalid_mask(self):
        for mask in [[], [True, False], [False]]:
            with self.subTest(mask=mask), self.assertRaises(ValueError):
                self.api.maxsim([[.1]], valid_doc=mask)

    def test_rrf_agreement(self):
        ranked = self.api.rrf([["A", "B"], ["C", "B"]])
        self.assertEqual([x[0] for x in ranked], ["B", "A", "C"])
        self.assertAlmostEqual(ranked[0][1], 2 / 62)

    def test_rrf_one_based(self):
        self.assertEqual(self.api.rrf([["B", "A"]], c=0), [("B", 1), ("A", .5)])

    def test_rrf_duplicate_rejection(self):
        with self.assertRaises(ValueError):
            self.api.rrf([["A", "A"]])

    def test_rrf_empty(self):
        self.assertEqual(self.api.rrf([[], []]), [])

    def test_rrf_invalid_c(self):
        for c in [-1, float("nan"), float("inf")]:
            with self.subTest(c=c), self.assertRaises(ValueError):
                self.api.rrf([["A"]], c=c)

    def test_recall_at_k_not_hit(self):
        self.assertEqual(self.api.recall_at_k({"A", "B", "C", "D"}, ["X", "A", "Y"], 3), .25)

    def test_recall_at_k_duplicate_positions(self):
        self.assertEqual(self.api.recall_at_k({"A", "B"}, ["A", "A", "B"], 2), .5)

    def test_recall_at_k_boundaries(self):
        self.assertEqual(self.api.recall_at_k({"A"}, ["A"], 0), 0)
        self.assertEqual(self.api.recall_at_k({"A"}, [], 9), 0)
        self.assertEqual(self.api.recall_at_k({"A"}, ["A"], 99), 1)

    def test_recall_at_k_undefined(self):
        with self.assertRaises(ValueError):
            self.api.recall_at_k(set(), ["A"], 1)

    def test_recall_at_k_invalid_k(self):
        for k in [-1, 1.5, True]:
            with self.subTest(k=k), self.assertRaises(ValueError):
                self.api.recall_at_k({"A"}, ["A"], k)

    @staticmethod
    def screening_example():
        return {"A": [[.95], [.05]], "B": [[.05], [.95]],
                "C": [[.80, 0], [0, .80]]}

    def test_candidate_union_misses_aggregate_winner(self):
        self.assertEqual(self.api.candidate_union(self.screening_example()), {"A", "B"})

    def test_candidate_union_wider_screening(self):
        self.assertEqual(self.api.candidate_union(self.screening_example(), per_query=2), {"A", "B", "C"})

    def test_candidate_union_tie_breaking(self):
        self.assertEqual(self.api.candidate_union({"B": [[.5]], "A": [[.5]]}), {"A"})

    def test_candidate_union_empty(self):
        self.assertEqual(self.api.candidate_union({}), set())

    def test_candidate_union_invalid(self):
        for k in [0, -1, 1.5, True]:
            with self.subTest(k=k), self.assertRaises(ValueError):
                self.api.candidate_union(self.screening_example(), per_query=k)
        for data in [{"A": [[.1]], "B": [[.2], [.3]]}, {"A": []}, {"A": [[float("inf")]]}]:
            with self.subTest(data=data), self.assertRaises(ValueError):
                self.api.candidate_union(data)


In [ ]:
import io
import math

NOTEBOOK_RESULTS = {}
EXERCISE_NAMES = ("tf_weight", "sparse_dot", "maxsim", "rrf", "recall_at_k", "candidate_union")

def check_exercise(name):
    if name not in EXERCISE_NAMES:
        raise ValueError("Unknown exercise: " + name)
    # Unselected functions use the reference only so unrelated TODOs do not interfere.
    functions = {n: (globals()[n] if n == name else getattr(REFERENCE, n)) for n in EXERCISE_NAMES}
    implementation = SimpleNamespace(**functions)
    checks = type("NotebookLearnerChecks", (RetrievalChecks,), {"api": implementation})
    names = [n for n in unittest.defaultTestLoader.getTestCaseNames(checks)
             if n.startswith("test_" + name + "_")]
    stream = io.StringIO()
    result = unittest.TextTestRunner(stream=stream, verbosity=2).run(
        unittest.TestSuite(checks(n) for n in names))
    if result.wasSuccessful():
        status = "PASS"
        print(f"PASS: {name} ({result.testsRun} contract tests)")
    elif result.errors and not result.failures and all("NotImplementedError" in tb for _, tb in result.errors):
        status = "TODO"
        print(f"TODO: implement {name}, rerun its definition, then rerun this check.")
    else:
        status = "FAIL"
        print(stream.getvalue())
    NOTEBOOK_RESULTS[name] = status
    return status

# Confirm that the reference implementation satisfies the same 30 checks.
_reference_result = unittest.TextTestRunner(verbosity=0).run(
    unittest.defaultTestLoader.loadTestsFromTestCase(RetrievalChecks))
assert _reference_result.wasSuccessful()
print("Reference ready. Your six functions below are still unfinished.")


## 1. BM25 weighting

Implement `tf * (k1 + 1) / (tf + k1 * (1 - b + b * length / avg_length))`, excluding IDF.

Require integer `0 <= tf <= length` (booleans are not integers for this contract), finite positive average length, finite nonnegative `k1`, and finite `b` in `[0,1]`. Raise `ValueError` otherwise. Zero TF returns zero, including when `k1=0`.

Predict the result for TF one and five at average document length with `k1=1.2`. Expected: 1.0 and approximately 1.7742. Why is the second result not five?

In [ ]:
def tf_weight(tf: int, length: int, avg_length: float, *,
              k1: float = 1.2, b: float = 0.75) -> float:
    """Exercise 1: BM25 length-adjusted TF. Validate the documented inputs."""
    raise NotImplementedError("Start with Exercise 1 in docs/exercises.md")


In [ ]:
check_exercise('tf_weight')


## 2. Sparse scoring

Multiply matching vocabulary coordinates and add the contributions. Missing keys have weight zero; empty inputs score zero. Inputs are dictionaries of finite numeric weights. No additional type validation is required.

For query `{car:1.5, automobile:0.7, repair:1.3, maintenance:0.9}` and document `{automobile:1.4, repair:0.8, maintenance:1.2}`, expect 3.10. These are constructed weights, not model outputs. Why does `car` not directly multiply `automobile`?

In [ ]:
def sparse_dot(query: Mapping[str, float], doc: Mapping[str, float]) -> float:
    """Exercise 2: multiply matching vocabulary coordinates, then sum."""
    raise NotImplementedError("Iterate over query coordinates; missing weights are zero")


In [ ]:
check_exercise('sparse_dot')


## 3. MaxSim and padding

Rows are query vectors; columns are document vectors. Exclude masked document columns, take each row's maximum, then sum.

Raise `ValueError` for empty/ragged/non-finite matrices, a mask of the wrong length, or a mask excluding every document position. Without a mask, all columns are valid.

`[[0.9,0.1,0],[0.1,0.8,0.2]]` scores 1.7. For `[[-0.2,-0.5,0]]` with mask `[True,True,False]`, expect -0.2, not zero. Why can zero padding beat valid evidence?

In [ ]:
def maxsim(matrix: Sequence[Sequence[float]], *,
           valid_doc: Sequence[bool] | None = None) -> float:
    """Exercise 3: rows=query, columns=document. Exclude padding before max."""
    raise NotImplementedError("Reduce document columns before summing query rows")


In [ ]:
check_exercise('maxsim')


## 4. Rank fusion

Each list contributes `1/(c + rank)` for each document it contains. Start ranks at one. Missing documents contribute zero. Default `c=60`.

Reject duplicates within a list and negative/non-finite `c`. Return `(ID, score)` pairs in descending score order, breaking ties alphabetically. Empty lists add nothing.

Fuse `[A,B]` and `[C,B]`. B should finish first with `2/62`. Does agreement establish relevance, or only consensus?

In [ ]:
def rrf(rankings: Sequence[Sequence[str]], *, c: float = 60.0) -> list[tuple[str, float]]:
    """Exercise 4: reciprocal ranks; reject duplicates within a list."""
    raise NotImplementedError("Ranks start at one; sort ties by document ID")


In [ ]:
check_exercise('rrf')


## 5. Recall, not hit

Divide unique relevant IDs found in the first `k` positions by the relevant-set size. Duplicates occupy positions but count once. Require a nonnegative integer `k`, excluding booleans. Raise `ValueError` if the relevant set is empty. `k=0` returns zero; `k` beyond the list length is allowed.

With relevant set `{A,B,C,D}` and ranking `[X,A,Y]`, Recall@3 is 0.25, while Hit@3 is one. Explain why deduplicating before slicing changes the task.

In [ ]:
def recall_at_k(relevant: set[str], ranking: Sequence[str], k: int) -> float:
    """Exercise 5: relevant-set recall, not the probability of at least one hit."""
    raise NotImplementedError("Count unique relevant IDs within the first k positions")


In [ ]:
check_exercise('recall_at_k')


## 6. Candidate screening

For each query row, score each document by its best token similarity. Keep the top `per_query` documents for that row, then union IDs. Ties are alphabetical.

Require positive integer `per_query` (not a boolean), finite rectangular nonempty matrices, and the same query-row count across documents. Empty maps return an empty set; asking for more candidates than documents is allowed. Raise `ValueError` on invalid input.

A has matrix `[[0.95],[0.05]]`; B has `[[0.05],[0.95]]`; C has `[[0.80,0],[0,0.80]]`. At `per_query=1`, expect `{A,B}`. Yet C wins the full MaxSim score. Explain why a better reranker cannot recover it from that candidate set. This toy procedure is not PLAID.

In [ ]:
def candidate_union(matrices: Mapping[str, Sequence[Sequence[float]]], *,
                    per_query: int = 1) -> set[str]:
    """Exercise 6: per-query-vector screening can miss the best MaxSim document."""
    raise NotImplementedError("Choose top documents per query row, then union IDs")


In [ ]:
check_exercise('candidate_union')


## After solving

Add one counterexample. Say which assumption it violates. Keep claims about constructed inputs separate from claims about trained models.

The key distinction is the score versus the candidate procedure. The score may be wrong, or the correct candidate may be absent before scoring.

Original sources: [BM25 parameter explanation](https://www.elastic.co/blog/practical-bm25-part-2-the-bm25-algorithm-and-its-variables), [SPLADE](https://arxiv.org/abs/2107.05720), [ColBERT](https://arxiv.org/abs/2004.12832), [RRF](https://cormack.uwaterloo.ca/cormacksigir09-rrf.pdf), and [PLAID](https://arxiv.org/abs/2205.09707).

Repository: `wangrui6/llm-engineering-lab`. Code and original teaching material: MIT license, copyright 2026 Rui Wang. Linked papers retain their own rights.

In [ ]:
print(NOTEBOOK_RESULTS)
